# 1. 导包
## 使用深度学习方法，这里选择用TextCNN来实现

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import jieba
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

d:\anaconda3\envs\AI_Env\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# 2. 数据预处理
- 主要是中文评论进行jieba分词
- 统计词频以构建词表
- 编码、对齐

In [2]:
df = pd.read_csv('Dataset/online_shopping_10_cats.csv').dropna(subset=['review', 'label'])
df['label'] = df['label'].astype(int)
df.head()

,cat,label,review
0,书籍,1,﻿做父母一定要有刘墉这样的心态，不断地学习，不断地进步，不断地给自己补充新鲜血液，让自己保持...
1,书籍,1,作者真有英国人严谨的风格，提出观点、进行论述论证，尽管本人对物理学了解不深，但是仍然能感受到...
2,书籍,1,作者长篇大论借用详细报告数据处理工作和计算结果支持其新观点。为什么荷兰曾经县有欧洲最高的生产...
3,书籍,1,作者在战几时之前用了＂拥抱＂令人叫绝．日本如果没有战败，就有会有美军的占领，没胡官僚主义的延...
4,书籍,1,作者在少年时即喜阅读，能看出他精读了无数经典，因而他有一个庞大的内心世界。他的作品最难能可贵...


In [3]:
# 分词
tokenized_texts = [list(jieba.cut(str(text))) for text in df['review']]

# 构建词表
all_words = [word for text in tokenized_texts for word in text]
# 取前一万个高频词，去除一些噪声
vocab = {word: i+2 for i, (word, _) in enumerate(Counter(all_words).most_common(10000))}
vocab['<PAD>'] = 0  
vocab['<UNK>'] = 1  

# 对文本编码，并且对齐
MAX_LEN = 50
def encode(text):
    encoded = [vocab.get(word, 1) for word in text][:MAX_LEN] # 切片去掉超过的
    return encoded + [0] * (MAX_LEN - len(encoded))

# 转换成tensor
X = torch.tensor([encode(text) for text in tokenized_texts], dtype=torch.long)
y = torch.tensor(df['label'].values, dtype=torch.long)

# 划分数据集并封装 DataLoder
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=128)

print(f"训练集大小: {len(X_train)}，测试集大小: {len(X_test)}")

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\lenovo\AppData\Local\Temp\jieba.cache
Loading model cost 0.612 seconds.
Prefix dict has been built successfully.


训练集大小: 50218，测试集大小: 12555


# 3. 定义网络
- TextCNN架构
    - embedding层：词嵌入，把前面算出的“一维数字 ID” 查表映射成“多维稠密向量”
    - 卷积层 (Conv2d) + 激活函数 (ReLU)：使用大小不同的卷积核（相当于滑动窗口），分别提取句子中 3个词、4个词、5个词 的局部语义特征（N-gram 特征）
    - 池化 + 全链接：提取特征最大值以及预测

In [4]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        # 词嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # 卷积层
        self.convs = nn.ModuleList([
            nn.Conv2d(in_channels=1, out_channels=100, kernel_size=(k, embed_dim)) 
            for k in (3, 4, 5)
        ])
        
        # 全连接层 (3个卷积核各出100维，拼起来300维)
        self.fc = nn.Linear(300, num_classes)

    def forward(self, x):
        # 增加一个Channel维度适应二维卷积
        x = self.embedding(x).unsqueeze(1) 
        
        # 卷积 -> 激活 -> 池化 -> 维度压缩
        x = [F.relu(conv(x)).squeeze(3) for conv in self.convs]
        x = [F.max_pool1d(i, i.size(2)).squeeze(2) for i in x]
        
        # 特征拼接并分类
        x = torch.cat(x, 1)
        return self.fc(x)

# 实例化模型
model = TextCNN(vocab_size=len(vocab), embed_dim=128, num_classes=2).to(device)

# 4. 模型训练与验证

In [6]:
NUM_EPOCHS = 20
best_acc = 0.0
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        out = model(batch_x)
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # 每轮跑完做一次验证
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            pred = model(batch_x).argmax(dim=1)
            correct += (pred == batch_y).sum().item()
    
    acc = correct / len(y_test)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Acc: {acc:.4f}")
    
    # 保存效果最好的模型
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(" ... 模型更新... ")

Epoch 1/20 | Loss: 0.0264 | Acc: 0.8996
 ... 模型更新... 
Epoch 2/20 | Loss: 0.0189 | Acc: 0.9038
 ... 模型更新... 
Epoch 3/20 | Loss: 0.0162 | Acc: 0.9068
 ... 模型更新... 
Epoch 4/20 | Loss: 0.0157 | Acc: 0.9086
 ... 模型更新... 
Epoch 5/20 | Loss: 0.0171 | Acc: 0.8995
Epoch 6/20 | Loss: 0.0140 | Acc: 0.9025
Epoch 7/20 | Loss: 0.0120 | Acc: 0.9081
Epoch 8/20 | Loss: 0.0164 | Acc: 0.9078
Epoch 9/20 | Loss: 0.0126 | Acc: 0.9032
Epoch 10/20 | Loss: 0.0123 | Acc: 0.9093
 ... 模型更新... 
Epoch 11/20 | Loss: 0.0138 | Acc: 0.9066
Epoch 12/20 | Loss: 0.0134 | Acc: 0.9068
Epoch 13/20 | Loss: 0.0152 | Acc: 0.9098
 ... 模型更新... 
Epoch 14/20 | Loss: 0.0131 | Acc: 0.9067
Epoch 15/20 | Loss: 0.0160 | Acc: 0.9106
 ... 模型更新... 
Epoch 16/20 | Loss: 0.0109 | Acc: 0.9126
 ... 模型更新... 
Epoch 17/20 | Loss: 0.0115 | Acc: 0.9087
Epoch 18/20 | Loss: 0.0127 | Acc: 0.9090
Epoch 19/20 | Loss: 0.0130 | Acc: 0.9051
Epoch 20/20 | Loss: 0.0107 | Acc: 0.9088


In [ ]:
# 加载模型验证并导出结果
# test_df = pd.read_csv('test.csv') # Kaggle判分的测试集还要处理，这里先从原本测试集划一个出来验证
# test_df['review'] = test_df['review'].fillna("")

test_results = []
model.load_state_dict(torch.load('best_model.pth')) # 加载表现最好的那次
model.eval()

# with torch.no_grad():
#     # 遍历测试集里的每一行
#     for index, row in test_df.iterrows():
#         item_id = row['id']
#         text = str(row['review'])
#         # 分词
#         words = list(jieba.cut(text))
#         # 编码对齐
#         encoded_text = encode(words) 
#         # 转tensor，维度对齐
#         input_tensor = torch.tensor([encoded_text], dtype=torch.long).to(device)
#         # 预测
#         pred = model(input_tensor).argmax(dim=1).item()
#         # 记录到结果列表里
#         test_results.append({'id': item_id, 'label': pred})

with torch.no_grad():
    for i in range(100):
        input_tensor = X_test[i].unsqueeze(0).to(device)
        pred = model(input_tensor).argmax(dim=1).item()
        test_results.append({'id': i, 'label': pred})

# 导出为 CSV
submission = pd.DataFrame(test_results)
submission.to_csv('submission.csv', index=False)